In [5]:
from pymodbus.client import ModbusTcpClient
import time as tt
import threading
client = ModbusTcpClient('210.119.14.74', port=502)
client.connect()

True

In [12]:
# ==================================================
# Vision Sensor All Numerical 값
# ==================================================

BLUE_PRODUCT_LID = 2
BLUE_PRODUCT_BASE = 3
GREEN_PRODUCT_LID = 5
GREEN_PRODUCT_BASE = 6

PRODUCT_NAMES = {
    BLUE_PRODUCT_LID: "Blue Product Lid",
    BLUE_PRODUCT_BASE: "Blue Product Base",
    GREEN_PRODUCT_LID: "Green Product Lid",
    GREEN_PRODUCT_BASE: "Green Product Base"
}


# ==================================================
# 센서 Input 주소
# ==================================================

# 분류용 Diffuse Sensor
GREEN_BASE_DIFFUSE = 0
BLUE_BASE_DIFFUSE = 1
GREEN_LID_DIFFUSE = 2
BLUE_LID_DIFFUSE = 3

# Machining Center 출구 Diffuse Sensor
MC0_EXIT_SENSOR = 4
MC1_EXIT_SENSOR = 5


# ==================================================
# Coil 주소
# ==================================================

# Emitter
EMITTER_RAW_0 = 21
EMITTER_RAW_1 = 22

# Pusher
BLUE_BASE_PUSHER = 31
BLUE_LID_PUSHER = 32
GREEN_BASE_PUSHER = 33
GREEN_LID_PUSHER = 34


# ==================================================
# 실제 분류 순서
#
# Vision
# → Blue Base
# → Blue Lid
# → Green Lid
# → Green Base
# ==================================================

STATIONS = [
    {
        "name": "Blue Base",
        "product": BLUE_PRODUCT_BASE,
        "diffuse": BLUE_BASE_DIFFUSE,
        "pusher": BLUE_BASE_PUSHER
    },
    {
        "name": "Blue Lid",
        "product": BLUE_PRODUCT_LID,
        "diffuse": BLUE_LID_DIFFUSE,
        "pusher": BLUE_LID_PUSHER
    },
    {
        "name": "Green Lid",
        "product": GREEN_PRODUCT_LID,
        "diffuse": GREEN_LID_DIFFUSE,
        "pusher": GREEN_LID_PUSHER
    },
    {
        "name": "Green Base",
        "product": GREEN_PRODUCT_BASE,
        "diffuse": GREEN_BASE_DIFFUSE,
        "pusher": GREEN_BASE_PUSHER
    }
]


# ==================================================
# 설정값
# ==================================================

EMITTER_ON_TIME = 0.3
PUSH_TIME = 0.7
TRACK_TIMEOUT = 30.0
LOOP_DELAY = 0.05


# 현재 작동 중인 출력
active_emitters = {}
active_pushers = {}


# ==================================================
# 기본 함수
# ==================================================

def coil(address, value):
    result = client.write_coil(address, value)

    if result is None or result.isError():
        print(f"Coil {address} 제어 실패")


def emit_one(emitter, now):
    """Emitter에서 원자재 한 개 생성"""

    if emitter in active_emitters:
        return

    coil(emitter, True)
    active_emitters[emitter] = now + EMITTER_ON_TIME

    print(f"Emitter Coil {emitter} → 원자재 생성")


def start_factory():
    """공장 가동"""

    # Coil 전체 초기화
    for address in range(35):
        coil(address, False)

    # Belt Conveyor 0~19 ON
    for address in range(20):
        coil(address, True)

    # Curved Conveyor 역방향 OFF
    coil(20, False)

    # Emitter는 필요할 때만 작동
    coil(EMITTER_RAW_0, False)
    coil(EMITTER_RAW_1, False)

    # Machining Center Stop 해제
    coil(25, False)
    coil(29, False)

    # Machining Center 0: Lid 생산
    coil(23, True)

    # Machining Center 1: Base 생산
    coil(27, False)

    # Machining Center Reset
    coil(26, True)
    coil(30, True)

    tt.sleep(1)

    coil(26, False)
    coil(30, False)

    # Machining Center Start 유지
    coil(24, True)
    coil(28, True)

    print("공장 자동 제어 시작")


def stop_factory():
    """전체 출력 정지"""

    for address in range(35):
        coil(address, False)

    client.close()
    print("공장 정지 및 Modbus 연결 종료")


# ==================================================
# 공장 시작
# ==================================================

start_factory()

# Factory I/O가 출력값을 받을 준비 시간
tt.sleep(0.5)


# 프로그램 시작 당시 센서 상태 읽기
initial_result = client.read_discrete_inputs(0, count=6)

if initial_result is not None and not initial_result.isError():
    previous_inputs = initial_result.bits[:6]
else:
    previous_inputs = [False] * 6


previous_vision = 0

# Vision Sensor를 통과한 물품 목록
tracked_products = []

next_product_id = 1


# ==================================================
# 최초 원자재 생성
# ==================================================

first_emit_time = tt.time()

emit_one(EMITTER_RAW_0, first_emit_time)
emit_one(EMITTER_RAW_1, first_emit_time)

print("최초 원자재 생성 완료")


# ==================================================
# 자동 제어 반복문
# ==================================================

try:
    while True:
        now = tt.time()

        # ------------------------------------------
        # 시간이 지난 Emitter OFF
        # ------------------------------------------

        for emitter, end_time in list(active_emitters.items()):
            if now >= end_time:
                coil(emitter, False)
                del active_emitters[emitter]

        # ------------------------------------------
        # 시간이 지난 Pusher OFF
        # ------------------------------------------

        for pusher, end_time in list(active_pushers.items()):
            if now >= end_time:
                coil(pusher, False)
                del active_pushers[pusher]

        # ------------------------------------------
        # Vision Sensor 읽기
        # Input Register 0
        # ------------------------------------------

        vision_result = client.read_input_registers(
            0,
            count=1
        )

        # ------------------------------------------
        # Diffuse Sensor Input 0~5 읽기
        # ------------------------------------------

        input_result = client.read_discrete_inputs(
            0,
            count=6
        )

        if (
            vision_result is None
            or input_result is None
            or vision_result.isError()
            or input_result.isError()
        ):
            print("센서 읽기 실패")
            tt.sleep(0.2)
            continue

        vision_value = vision_result.registers[0]
        inputs = input_result.bits[:6]

        # ==========================================
        # MC0 출구 감지 → Emitter-Raw 0 생성
        # ==========================================

        mc0_exit_detected = (
            inputs[MC0_EXIT_SENSOR]
            and not previous_inputs[MC0_EXIT_SENSOR]
        )

        if mc0_exit_detected:
            print("MC0 가공품 출구 도착 → 다음 원자재 생성")
            emit_one(EMITTER_RAW_0, now)

        # ==========================================
        # MC1 출구 감지 → Emitter-Raw 1 생성
        # ==========================================

        mc1_exit_detected = (
            inputs[MC1_EXIT_SENSOR]
            and not previous_inputs[MC1_EXIT_SENSOR]
        )

        if mc1_exit_detected:
            print("MC1 가공품 출구 도착 → 다음 원자재 생성")
            emit_one(EMITTER_RAW_1, now)

        # ==========================================
        # Vision Sensor 신규 물품 등록
        # ==========================================

        vision_detected = (
            vision_value != 0
            and vision_value != previous_vision
        )

        if vision_detected:

            if vision_value in PRODUCT_NAMES:

                tracked_products.append({
                    "id": next_product_id,
                    "type": vision_value,
                    "last_station": -1,
                    "detected_time": now
                })

                print(
                    f"[물품 {next_product_id}] "
                    f"Vision 판별: {PRODUCT_NAMES[vision_value]}"
                )

                next_product_id += 1

            else:
                print(f"분류 대상이 아닌 Vision 값: {vision_value}")

        previous_vision = vision_value

        # ==========================================
        # 추적 시간이 지난 물품 제거
        # ==========================================

        for product in tracked_products.copy():

            if now - product["detected_time"] > TRACK_TIMEOUT:

                print(
                    f"[물품 {product['id']}] "
                    f"추적 시간 초과"
                )

                tracked_products.remove(product)

        # ==========================================
        # Diffuse Sensor와 Pusher 분류
        # ==========================================

        for station_index, station in enumerate(STATIONS):

            diffuse_address = station["diffuse"]

            diffuse_detected = (
                inputs[diffuse_address]
                and not previous_inputs[diffuse_address]
            )

            if not diffuse_detected:
                continue

            # 현재 분류 위치에 도착할 물품 검색
            target_product = None

            for product in tracked_products:

                if product["last_station"] == station_index - 1:
                    target_product = product
                    break

            if target_product is None:

                print(
                    f"{station['name']} 센서 감지 "
                    f"→ 추적 중인 물품 없음"
                )

                continue

            product_name = PRODUCT_NAMES[target_product["type"]]

            # Vision 판별 결과와 분류 위치가 일치
            if target_product["type"] == station["product"]:

                pusher = station["pusher"]

                if pusher not in active_pushers:

                    coil(pusher, True)
                    active_pushers[pusher] = now + PUSH_TIME

                print(
                    f"[물품 {target_product['id']}] "
                    f"{product_name} "
                    f"→ {station['name']} Pusher 작동"
                )

                # 분류 완료
                tracked_products.remove(target_product)

            else:
                # 현재 분류 대상이 아니므로 다음 지점으로 이동
                target_product["last_station"] = station_index

                print(
                    f"[물품 {target_product['id']}] "
                    f"{product_name} "
                    f"→ {station['name']} 통과"
                )

                # 마지막 분류 지점도 통과
                if station_index == len(STATIONS) - 1:

                    print(
                        f"[물품 {target_product['id']}] "
                        f"분류 실패"
                    )

                    tracked_products.remove(target_product)

        previous_inputs = inputs.copy()

        tt.sleep(LOOP_DELAY)


except KeyboardInterrupt:
    print("\n사용자가 자동 제어를 중지했습니다.")


finally:
    stop_factory()

공장 자동 제어 시작
Emitter Coil 21 → 원자재 생성
Emitter Coil 22 → 원자재 생성
최초 원자재 생성 완료
MC1 가공품 출구 도착 → 다음 원자재 생성
Emitter Coil 22 → 원자재 생성
MC0 가공품 출구 도착 → 다음 원자재 생성
Emitter Coil 21 → 원자재 생성
[물품 1] Vision 판별: Blue Product Base
[물품 1] Blue Product Base → Blue Base Pusher 작동
MC1 가공품 출구 도착 → 다음 원자재 생성
Emitter Coil 22 → 원자재 생성

사용자가 자동 제어를 중지했습니다.
공장 정지 및 Modbus 연결 종료
